# Backtest EMACross

Run the EMACross strategy on a simulated exchange venue.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — fast/slow grid with heatmap

**Prerequisites:** Run `scripts/fetch_binance_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.backtesting.engine import resolve_strategy_liquidation_config
from src.core import (
    TOPIC_ACCOUNT_LIQUIDATED,
    TOPIC_POSITION_LIQUIDATED,
    LiquidationConfig,
    SizingConfig,
    bar_type_str,
    get_venue_config,
    with_venue_config,
)
from src.strategies.ma_cross import MACross, MACrossConfig

import sys
sys.path.insert(0, str(__import__("pathlib").Path("..").resolve()))

from charts import plot_ma_cross, plot_equity_curve, print_summary_stats, plot_pnl_heatmap, generate_backtest_html
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────
DATA_SOURCE      = "BINANCE_PERP"       # where the catalog data comes from
EXEC_VENUE       = "HYPERLIQUID_PERP"   # exchange we're simulating execution on
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
DATE_START       = None                 # e.g. "2025-01-01" — None means "from start of data"
DATE_END         = None                 # e.g. "2025-04-01" — None means "to end of data"
MA_TYPE          = "EMA"                # strategy variant — do not change
STARTING_CAPITAL = 1000
TRADE_SIZE       = 2000                  # $100 USD notional per trade (back-compat path)
LEVERAGE         = 20
SAVE_TEARSHEET   = False

# MA params
FAST_MA  = 10
SLOW_MA  = 40

# Sweep grids
FAST_PERIODS = sorted(set([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100] + [FAST_MA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_MA]))

# ── Sizing (Project A) ────────────────────────────────────────—
# None → fall back to TRADE_SIZE/trade_notional via SizingConfig(mode="fixed").
# To exercise equity-fraction sizing instead:
#   SIZING = SizingConfig(
#       mode="equity_frac",
#       risk_frac=Decimal("0.10"),    # 10% of equity at risk per trade
#       stop_pct=Decimal("0.05"),     # 5% protective stop
#       min_notional=Decimal("50"),
#   )
SIZING: SizingConfig | None = None

# ── Liquidation simulator (Project B) ────────────────────────—
# enabled=True wires up:
#   - LiquidationAware mixin (places a reduce-only stop at the cross-margin
#     liq price for every position), AND
#   - AccountAliveMonitor actor (halts the RiskEngine when equity falls
#     below the floor required to open a min-size order).
#
# With STARTING_CAPITAL=1000 and TRADE_SIZE=100, gross leverage is 0.1× —
# this config will NOT trigger any liquidations. To exercise the full
# path, raise TRADE_SIZE (e.g. 2000 → 2× gross leverage → liq at ~49.5%
# adverse) or switch SIZING above to equity_frac mode.
#
# Set enabled=False to run the legacy path with no liquidation simulation
# (back-compat with old sweep parquets).
LIQUIDATION = LiquidationConfig(
    enabled=True,
    halt_on_account_liquidation=True,
    # mm_rate, fee_rate, min_trade_notional all None — resolved at engine
    # build time from VENUE_CFG (mm_rate, taker_fee) and SIZING/instrument.
)

# Derived values
DATA_CFG       = get_venue_config(DATA_SOURCE) # Resolve venue configs —
VENUE_CFG      = get_venue_config(EXEC_VENUE)  # fees and leverage come from the registry
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, DATA_SOURCE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(DATA_CFG.nt_venue)

# Files
CATALOG_PATH = "../../data/catalog"
SWEEP_NAME   = f"MACross-{MA_TYPE}_{ASSET}_{EXEC_VENUE}_{BAR_INTERVAL}"
RESULT_NAME  = (
    f"MACross-{MA_TYPE}_{ASSET}_{EXEC_VENUE}_{BAR_INTERVAL}"
    f"_f{FAST_MA}_s{SLOW_MA}"
)

print("Imports OK")
print(f"Sizing       : {'equity_frac' if SIZING and SIZING.mode == 'equity_frac' else 'fixed (trade_notional back-compat)'}")
print(f"Liquidation  : {'ENABLED' if LIQUIDATION.enabled else 'disabled'}")

In [ ]:
# ── Cell 2: Load data + resolve liquidation config ────────────────

catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

if DATE_START or DATE_END:
    start_ns = pd.Timestamp(DATE_START, tz="UTC").value if DATE_START else None
    end_ns   = pd.Timestamp(DATE_END,   tz="UTC").value if DATE_END   else None
    bars = [
        b for b in bars
        if (start_ns is None or b.ts_event >= start_ns)
        and (end_ns   is None or b.ts_event <= end_ns)
    ]

# Override fees with the simulated exec venue's config
# Leverage is applied at the venue level in make_engine(), not here.
instrument = with_venue_config(
    instrument,
    maker_fee=VENUE_CFG.maker_fee,
    taker_fee=VENUE_CFG.taker_fee,
)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

# Synthesize a sizing config for liquidation resolution when none is set.
# The strategy still uses SIZING (None → back-compat trade_notional path);
# this synthetic value only seeds the resolver's min_trade_notional fallback
# so resolve_strategy_liquidation_config doesn't raise when the instrument
# has no min_notional set.
_sizing_for_resolution = SIZING or SizingConfig(
    mode="fixed",
    fixed_notional=TRADE_NOTIONAL,
)

# Resolve LIQUIDATION fields (mm_rate, fee_rate, min_trade_notional)
# from VENUE_CFG / SIZING / instrument so the strategy mixin sees a
# fully-resolved LiquidationConfig.  make_engine resolves independently
# for the AccountAliveMonitor; passing LIQ_RESOLVED to both keeps them
# in sync.
LIQ_RESOLVED = resolve_strategy_liquidation_config(
    user=LIQUIDATION,
    venue_config=VENUE_CFG,
    instrument=instrument,
    sizing=_sizing_for_resolution,
)

print(f"Data source : {DATA_SOURCE}")
print(f"Venue       : {VENUE}")
print(f"Exec venue  : {EXEC_VENUE} (simulated)")
print(f"Instrument  : {instrument.id}")
print(f"Currency    : {CURRENCY}")
print(f"Leverage    : {LEVERAGE}x")
print(f"Maker fee   : {instrument.maker_fee}  (from {EXEC_VENUE})")
print(f"Taker fee   : {instrument.taker_fee}  (from {EXEC_VENUE})")
print(f"Bar count   : {len(bars):,}")
print(f"First bar   : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar    : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")
if DATA_SOURCE != EXEC_VENUE:
    print(f"⚠ Cross-venue simulation: {DATA_SOURCE} data → {EXEC_VENUE} fees")

# Resolved liquidation values — confirms what the simulator will use.
if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
    floor_im   = float(LIQ_RESOLVED.min_trade_notional) / LEVERAGE
    fee_buffer = (
        float(LIQ_RESOLVED.min_trade_notional)
        * float(LIQ_RESOLVED.fee_rate)
        * 2
        * LIQ_RESOLVED.alive_trades_buffer
    )
    print()
    print(f"Liquidation        : ENABLED")
    print(f"  mm_rate           : {LIQ_RESOLVED.mm_rate}")
    print(f"  fee_rate          : {LIQ_RESOLVED.fee_rate}")
    print(f"  min_trade_notional: {LIQ_RESOLVED.min_trade_notional}")
    print(f"  alive_buffer      : {LIQ_RESOLVED.alive_trades_buffer}")
    print(f"  halt on dead acct : {LIQ_RESOLVED.halt_on_account_liquidation}")
    print(f"  alive threshold   : equity ≥ ${floor_im + fee_buffer:.4f}  "
          f"(IM=${floor_im:.4f} + fees=${fee_buffer:.4f})")
else:
    print()
    print("Liquidation        : disabled (set LIQUIDATION.enabled=True in Cell 1)")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = make_engine(
    VENUE, instrument, bars, STARTING_CAPITAL,
    leverage=LEVERAGE,
    liquidation=LIQ_RESOLVED,
    venue_config=VENUE_CFG,
    sizing=SIZING,
)
print("Engine configured.")

# Confirm AccountAliveMonitor wiring when liquidation is enabled.
if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
    actor_ids = [str(a) for a in engine.kernel.trader.actor_ids()]
    aam = [a for a in actor_ids if "AccountAlive" in a]
    print(f"  Registered actors  : {actor_ids}")
    print(f"  AccountAliveMonitor: {'✓ present' if aam else '✗ MISSING'}")
    print(f"  RiskEngine state   : {engine.kernel.risk_engine.trading_state}")

In [ ]:
# ── Cell 3b: Subscribe to liquidation events for diagnostics ─────
#
# The mixin publishes on TOPIC_POSITION_LIQUIDATED whenever a liquidation
# stop fills; AccountAliveMonitor publishes on TOPIC_ACCOUNT_LIQUIDATED
# when the alive predicate flips false.  Capture both for inspection
# after engine.run().

position_liquidations: list = []
account_liquidations: list = []

engine.kernel.msgbus.subscribe(
    topic=TOPIC_POSITION_LIQUIDATED,
    handler=position_liquidations.append,
)
engine.kernel.msgbus.subscribe(
    topic=TOPIC_ACCOUNT_LIQUIDATED,
    handler=account_liquidations.append,
)
print("Subscribed to liquidation topics — events will accumulate during run.")

In [ ]:
# ── Cell 4: Add MACross strategy + run ────────────────────────────
config = MACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    sizing=SIZING,
    trade_notional=TRADE_NOTIONAL,
    ma_type=MA_TYPE,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    liquidation=LIQ_RESOLVED,
)
strategy = MACross(config=config)

# Confirm the strategy received the resolved configs.
_s = strategy._sizing
print(f"Strategy._sizing      : mode={_s.mode}, fixed={_s.fixed_notional}, "
      f"risk_frac={_s.risk_frac}, stop_pct={_s.stop_pct}, "
      f"min_notional={_s.min_notional}")
print(f"Strategy._liq_config  : {strategy._liq_config}")

engine.add_strategy(strategy)
engine.run()
print()
print("Backtest complete.")

# Post-run state confirmation.
print(f"Post-run RiskEngine state    : {engine.kernel.risk_engine.trading_state}")
print(f"Position liquidations fired  : {len(position_liquidations)}")
print(f"Account liquidations fired   : {len(account_liquidations)}")
if position_liquidations:
    print()
    print("First 5 position liquidation events:")
    for ev in position_liquidations[:5]:
        print(f"  {ev}")
if account_liquidations:
    print()
    print("Account liquidation event:")
    for ev in account_liquidations:
        print(f"  {ev}")

In [ ]:
# ── Cell 5: Reports ──────────────────────────────────────────────—
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")

# Min balance + cross-check with liquidation events.
if not account_report.empty:
    totals = account_report["total"].astype(float)
    min_bal = totals.min()
    print(f"Min balance : {min_bal:.4f}")

    if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
        # With AccountAliveMonitor running, min_balance should not go far
        # below the configured floor. If it does, the actor missed a beat.
        if min_bal < -1.0 and not account_liquidations:
            print(f"⚠ min_balance={min_bal:.2f} but no AccountLiquidated event "
                  f"fired — actor missed an equity breach.")
        if account_liquidations and min_bal <= 0:
            print(f"  (min_bal ≤ 0 after AccountLiquidated is expected from "
                  f"residual fee debits.)")
    else:
        if min_bal <= 0:
            print(f"\n⚠ LIQUIDATED — min balance was {min_bal:.2f}")
            print("PnL results after liquidation are meaningless.")

print()
print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 5b:  ──────────────────────────────────

orders_report = engine.trader.generate_orders_report()

print(f"Total orders: {len(orders_report)}")
print(orders_report["status"].value_counts())

# Filter to just the denied/rejected ones
denied = orders_report[orders_report["status"].isin(["DENIED", "REJECTED"])]
print(f"Denied/rejected: {len(denied)}")
if not denied.empty:
    display(denied[["ts_init", "side", "quantity", "status"]])

status_counts = orders_report["status"].value_counts()
print("Order status breakdown:")
print(status_counts)

denied_count = orders_report[
    orders_report["status"].isin(["DENIED", "REJECTED"])
].shape[0]
if denied_count > 0:
    print(f"\n⚠ {denied_count} orders were denied or rejected")
else:
    print(f"\n✓ All {len(orders_report)} orders filled or are open")


# Diagnostic: compare single-run vs sweep liquidation detection
acct_report_check = engine.trader.generate_account_report(VENUE)
print(f"Account report rows: {len(acct_report_check)}")
print(f"Columns: {list(acct_report_check.columns)}")
print(f"Dtypes: {dict(acct_report_check.dtypes)}")

if not acct_report_check.empty:
    # Cast to float for numerical ops (column is object/string to preserve precision)
    totals = acct_report_check["total"].astype(float)
    
    min_bal   = totals.min()
    max_bal   = totals.max()
    first_bal = totals.iloc[0]
    final_bal = totals.iloc[-1]
    
    print(f"First balance: {first_bal:.4f}")
    print(f"Min balance:   {min_bal:.4f}")
    print(f"Max balance:   {max_bal:.4f}")
    print(f"Final balance: {final_bal:.4f}")
    print(f"Would flag liquidated: {min_bal <= 0}")
    
    # Copy with float total for nsmallest
    tmp = acct_report_check.copy()
    tmp["total_float"] = totals
    print("\n5 lowest balance rows:")
    print(tmp.nsmallest(5, "total_float")[["total", "free", "locked"]])

totals = account_report["total"].astype(float)
min_bal = totals.min()
if min_bal <= 0:
    print(f"\n⚠ LIQUIDATED — min balance was {min_bal:.2f}")
    print("PnL results after liquidation are meaningless.")


# Direct calculation — what does NT do with these field values?
notional = Decimal("2000")
leverage = 20

# If "percentage of order value" (CryptoPerpetual doc):
im_pct = float(instrument.margin_init) * float(notional)
mm_pct = float(instrument.margin_maint) * float(notional)

# If MarginAccount formula (notional/leverage × field):
im_formula = (float(notional) / leverage) * float(instrument.margin_init)
mm_formula = (float(notional) / leverage) * float(instrument.margin_maint)

print(f"margin_init field:  {instrument.margin_init}")
print(f"margin_maint field: {instrument.margin_maint}")
print()
print(f"If 'pct of order value':   IM=${im_pct:.2f}  MM=${mm_pct:.2f}")
print(f"If 'notional/lev × field': IM=${im_formula:.2f}  MM=${mm_formula:.2f}")

print("First 5 account report rows:")
print(account_report[["total", "free", "locked"]].head(5).to_string())


In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=(
        f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA}) | {ASSET} {BAR_INTERVAL}"
        f" | data: {DATA_SOURCE} | sim: {EXEC_VENUE}"
    ),
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Price chart with MA overlay + trade markers ──────────—

fig = plot_ma_cross(
    bars, fills_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────
plot_equity_curve(analyzer, account_report, f"{MA_TYPE}Cross({FAST_MA}/{SLOW_MA})  {INSTRUMENT_ID} {BAR_INTERVAL}")

In [ ]:
# ── Cell 10: Summary statistics ────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep ──────────────────────────────────────
#
# Grid search over fast/slow MA period combinations.
# Liquidation simulation runs per-engine when LIQ_RESOLVED is enabled.

def ma_factory(eng, params):
    cfg = MACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        sizing=SIZING,
        trade_notional=TRADE_NOTIONAL,
        ma_type=MA_TYPE,
        fast_period=params["fast"],
        slow_period=params["slow"],
        liquidation=LIQ_RESOLVED,
    )
    eng.add_strategy(MACross(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ma_factory,
    strategy_name=f"MACross-{MA_TYPE}-{EXEC_VENUE}",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    sweep_name=SWEEP_NAME,
    leverage=LEVERAGE,
    liquidation=LIQ_RESOLVED,
    venue_config=VENUE_CFG,
    sizing=SIZING,
)

In [ ]:
# ── Cell 12: PnL heatmap ──────────────────────────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label=f"Slow {MA_TYPE} Period", col_label=f"Fast {MA_TYPE} Period",
    title=f"Total PnL ({CURRENCY}) by {MA_TYPE} Parameters",
)

In [ ]:
# ── Cell 12b: Sweep liquidation diagnostics ──────────────────────
#
# Trustworthiness checks for the liquidation simulator and fee model:
#
# 1. Schema completeness — every row has populated liq columns.
# 2. Min balance / liquidated_account consistency — if equity went
#    below zero but the actor didn't fire, the actor missed a breach.
# 3. Halt enforcement — for combos with liquidated_account=True, we
#    expect denied_post_halt > 0 (the strategy keeps signaling but
#    RiskEngine HALTED rejects the submits).
# 4. Fee model cross-check — total_fees / num_positions should be
#    roughly 2 × avg_notional × taker_fee (round-trip per position).
# 5. Liquidation slippage — fill price vs trigger price.  Tells us
#    whether bar decomposition fills cleanly at the trigger or gaps
#    through.  Sign convention: positive % = worse than trigger
#    (gap-risk loss).

if LIQ_RESOLVED is not None and LIQ_RESOLVED.enabled:
    cols = [
        "fast", "slow",
        "liquidated_positions", "liquidated_account", "liquidated_at_ts",
        "denied_post_halt",
        "liq_slippage_avg_pct", "liq_slippage_max_pct",
        "min_balance", "final_balance", "total_pnl",
        "total_fees",
    ]
    available = [c for c in cols if c in results_df.columns]

    n_pos_liq  = int((results_df["liquidated_positions"] > 0).sum())
    n_acct_liq = int(results_df["liquidated_account"].sum())
    n_negbal   = int((results_df["min_balance"] < 0).sum())
    n_denied   = int((results_df["denied_post_halt"] > 0).sum())

    print("=== Liquidation summary ===")
    print(f"Total combos          : {len(results_df)}")
    print(f"With position liq     : {n_pos_liq}")
    print(f"With account liq      : {n_acct_liq}")
    print(f"With denied post-halt : {n_denied}")
    print(f"Sub-zero min_balance  : {n_negbal}")

    if n_pos_liq > 0:
        print()
        print("Top 10 by liquidated_positions:")
        display(results_df.nlargest(10, "liquidated_positions")[available])

    # Sanity check 1: any row with min_balance ≤ 0 should also have
    # liquidated_account=True, otherwise the actor missed the breach.
    inconsistent = results_df[
        (results_df["min_balance"] <= 0)
        & (~results_df["liquidated_account"].astype(bool))
    ]
    if len(inconsistent) > 0:
        print()
        print(f"⚠ {len(inconsistent)} rows with min_balance ≤ 0 but "
              f"liquidated_account=False — actor missed equity breach.")
        display(inconsistent[available])
    else:
        print()
        print("✓ min_balance / liquidated_account consistent across all rows")

    # Sanity check 2: dead combos should have denied_post_halt > 0
    halt_no_denials = results_df[
        results_df["liquidated_account"].astype(bool)
        & (results_df["denied_post_halt"] == 0)
    ]
    if len(halt_no_denials) > 0:
        print()
        print(f"ℹ {len(halt_no_denials)} dead combos with no post-halt denials "
              f"(strategy didn't re-signal after halt — usually fine).")
        display(halt_no_denials[available])
    else:
        print("✓ Every dead combo had at least one post-halt denial — "
              "HALTED state is enforcing.")

    # Sanity check 3: fee model.  Expected round-trip fee per position:
    #   2 × notional × taker_fee.  Survivors only (dead combos have fewer trades).
    survivors = results_df[~results_df["liquidated_account"].astype(bool)]
    if not survivors.empty:
        avg_fee_per_position = (
            survivors["total_fees"] / survivors["num_positions"]
        ).replace([float("inf"), -float("inf")], float("nan")).mean()
        expected = 2 * float(TRADE_NOTIONAL) * float(LIQ_RESOLVED.fee_rate)
        ratio = avg_fee_per_position / expected if expected > 0 else float("nan")
        # 10% tolerance allows for price drift between qty calc and fill on a
        # trending asset over a multi-year backtest.
        print()
        print(f"=== Fee model cross-check (survivors) ===")
        print(f"Avg fees per position : ${avg_fee_per_position:.4f}")
        print(f"Expected round-trip   : ${expected:.4f}  "
              f"(2 × ${float(TRADE_NOTIONAL):.0f} × {float(LIQ_RESOLVED.fee_rate):.5f})")
        print(f"Ratio actual/expected : {ratio:.3f}  "
              f"({'✓ within 10%' if 0.90 <= ratio <= 1.10 else '⚠ outside 10%'})")
        if 1.02 <= ratio <= 1.10:
            print(f"  (Price-drift between qty calc and fill on a trending "
                  f"asset typically pushes ratio 2-8% above 1.0.)")

    # Sanity check 4: liquidation-stop slippage.  Per-event detail across
    # the dead combos.  trigger_price = where the mixin asked the stop
    # to fire; fill_price = where the matching engine actually filled it.
    # In NT bar decomposition, the fill price is the synthetic-tick price
    # that crossed the trigger — could match (clean fill) or gap below
    # (worse fill = positive slippage %).
    liq_rows = results_df[results_df["liquidated_positions"] > 0]
    if not liq_rows.empty:
        print()
        print("=== Liquidation slippage (trigger vs fill, % of entry) ===")
        print(f"Combos with liquidations    : {len(liq_rows)}")
        print(f"Avg slippage across combos  : {liq_rows['liq_slippage_avg_pct'].mean():.4f}%")
        print(f"Worst single-event slippage : {liq_rows['liq_slippage_max_pct'].max():.4f}%")
        # Distribution
        clean_fills = (liq_rows["liq_slippage_max_pct"].abs() < 0.01).sum()
        print(f"Combos with clean fills (|slippage|<0.01%) : {clean_fills}/{len(liq_rows)}")
        if liq_rows["liq_slippage_max_pct"].max() > 1.0:
            worst = liq_rows.nlargest(5, "liq_slippage_max_pct")[
                ["fast", "slow", "liq_slippage_avg_pct", "liq_slippage_max_pct",
                 "liquidated_at_ts", "total_pnl"]
            ]
            print()
            print("Top 5 worst slippage events (gap risk surfaced):")
            display(worst)
else:
    print("Liquidation simulation off — new columns will be 0/False/None for all rows.")

In [ ]:
# -- Cell 13: TradingView Interactive Backtest ------------------

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    ma_type=MA_TYPE,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
    open_browser=True,
)

In [ ]:
# ── Cell 14: Save notebook snapshot ────────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("ema_cross.ipynb", RESULT_NAME)
#save_notebook_html("ema_cross.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 15: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")
